In [ ]:
# ==============================================================================
# STEP 1: IMPORTS & DTMF CONFIGURATION
# ==============================================================================

import json
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

# Standard DTMF (Dual-Tone Multi-Frequency) signaling frequencies:
# Low group (rows):     697 Hz, 770 Hz, 852 Hz, 941 Hz
# High group (columns): 1209 Hz, 1336 Hz, 1477 Hz
FREQS = np.array([697, 770, 852, 941, 1209, 1336, 1477], dtype=float)

# Standard 4x3 telephone keypad layout mapping row & column intersections to keys
KEYPAD = np.array([
    ["1", "2", "3"],
    ["4", "5", "6"],
    ["7", "8", "9"],
    ["*", "0", "#"]
])
# Flattened keypad array: index = (row_idx * 3) + col_idx (maps 0..11 to characters)
KEYS_FLAT = KEYPAD.ravel()

# Human-readable labels for the primary IVR main menu options
MAIN_NAMES = {"1": "Balance", "2": "Transfers", "3": "Cards", "4": "Loans", "0": "Agent"}

# Spectrogram frame analysis parameters:
# 30 ms window provides adequate frequency resolution (~33 Hz bin spacing) to resolve 70-80 Hz DTMF gaps
# 10 ms hop provides 20 ms overlap for smooth temporal tracking of button presses
FRAME_MS, HOP_MS = 30, 10


def tone_strengths(x, fs, frame_ms=30, hop_ms=10):
    """
    Compute frame-level spectral energy for each of the 7 standard DTMF frequencies.

    Parameters:
        x (np.ndarray): 1D audio waveform array (int16 or float64).
        fs (int): Sampling rate of the audio in Hertz (Hz).
        frame_ms (int): Duration of each analysis frame in milliseconds (default 30 ms).
        hop_ms (int): Step size between consecutive frames in milliseconds (default 10 ms).

    Returns:
        np.ndarray: 2D array of shape (num_frames, 7) containing the spectral magnitudes
                    at [697, 770, 852, 941, 1209, 1336, 1477] Hz.
    """
    freqs = np.array([697, 770, 852, 941, 1209, 1336, 1477])

    # Step 1.1: Convert millisecond time parameters to discrete sample counts
    n = int(fs * frame_ms / 1000)      # Window length in samples
    hop = int(fs * hop_ms / 1000)     # Hop / stride length in samples

    # Step 1.2: Slice the audio signal into overlapping frames using sliding window view
    frames = sliding_window_view(x, n)[::hop]

    # Step 1.3: Apply a Hanning window to taper frame edges and reduce spectral leakage
    frames = frames * np.hanning(n)

    # Step 1.4: Compute the Real FFT (rfft) along each frame to obtain magnitude spectrum
    spectrum = np.abs(np.fft.rfft(frames, axis=1))

    # Step 1.5: Determine FFT frequency bin centers
    bins = np.fft.rfftfreq(n, 1 / fs)

    # Step 1.6: Find the nearest FFT bin index for each target DTMF frequency
    dist = np.abs(bins[None, :] - freqs[:, None])

    # Step 1.7: Extract spectral magnitudes at target DTMF bins for all frames
    return spectrum[:, dist.argmin(axis=1)]



# Data and Structure

In [ ]:
# ==============================================================================
# STEP 2: DATA LOADING, VALIDATION & CALL SEGMENTATION
# ==============================================================================

def load_data(path):
    """
    Read the IVR .npz archive, extract raw fields, and validate data integrity.

    Parameters:
        path (str): Filepath to the .npz archive (e.g., 'group04_ivr.npz').

    Returns:
        tuple: (audio, offsets, fs, menu)
            - audio (np.ndarray): 1D continuous audio stream of concatenated calls.
            - offsets (np.ndarray): 1D int64 sample boundaries of shape (num_calls + 1,).
            - fs (np.ndarray): 1D int64 sample rates per call of shape (num_calls,).
            - menu (dict): Parsed JSON dictionary detailing IVR menus and options.
    """
    # Step 2.1: Load the numpy zipped archive (.npz)
    d = np.load(path)

    # Step 2.2: Extract audio stream, sample offsets, and sampling rates
    audio = d["audio"]
    offsets = d["call_offsets"].astype(np.int64)
    fs = d["sample_rate_per_call"].astype(np.int64)

    # Step 2.3: Deserialize menu structure JSON string (handles byte/string encoding)
    raw = np.asarray(d["menu_json"]).item()
    menu = json.loads(raw.decode() if isinstance(raw, bytes) else str(raw))

    # Step 2.4: Validate structural consistency and data assertions
    assert audio.ndim == 1, audio.shape  # Audio must be a single 1D continuous stream
    assert offsets.shape == (fs.size + 1,), (offsets.shape, fs.size)  # N calls require N+1 boundary offsets
    assert offsets[0] == 0 and offsets[-1] <= audio.size  # First call starts at sample 0
    assert np.all(np.diff(offsets) > 0), "call offsets must be strictly increasing"  # No zero-length calls

    return audio, offsets, fs, menu


def split_calls(audio, offsets):
    """
    Segment the contiguous concatenated audio stream into individual call recordings.

    Parameters:
        audio (np.ndarray): Full concatenated 1D audio array.
        offsets (np.ndarray): Array of call boundary sample indices.

    Returns:
        list of np.ndarray: List of audio segments, one per call.
    """
    # Step 2.5: Slice audio into discrete per-call arrays using split offsets
    calls = np.split(audio[:offsets[-1]], offsets[1:-1])

    # Step 2.6: Verify segmentation guarantees
    assert len(calls) == offsets.size - 1  # Exactly n_calls extracted
    assert sum(c.size for c in calls) == offsets[-1]  # All audio samples accounted for

    return calls



# Data Problems

In [ ]:
# ==============================================================================
# STEP 3: SIGNAL QUALITY DIAGNOSTICS & ABLATION STUDY
# ==============================================================================

def diagnose(calls, fs):
    """
    Detect and flag three common recording anomalies across all calls:
      1. Odd Sample Rates: Discrepancies from the modal/dominant sample rate.
      2. DC Offset / Baseline Shift: Constant bias shifting the audio away from zero.
      3. Amplitude Clipping: Saturation hitting the minimum/maximum int16 boundaries.

    Parameters:
        calls (list of np.ndarray): Raw audio arrays for each call.
        fs (np.ndarray): Sample rate for each call.

    Returns:
        dict: Diagnostic metrics containing modal_fs, odd_rate, median, std, clip_frac,
              offset_calls, and clipped_calls indices.
    """
    # Step 3.1: Find the dominant (modal) sampling rate across all recordings
    vals, counts = np.unique(fs, return_counts=True)
    modal_fs = int(vals[counts.argmax()])

    # Step 3.2: Compute per-call statistics: baseline offset (median) and dispersion (std)
    med = np.array([np.median(c) for c in calls])
    sd = np.array([c.astype(np.float64).std() for c in calls])

    # Step 3.3: Calculate fraction of samples pinned at int16 extremes (clipping saturation)
    clip = np.array([np.mean((c == c.max()) | (c == c.min())) for c in calls])

    # Step 3.4: Verify output dimensions match number of calls
    assert med.shape == sd.shape == clip.shape == fs.shape

    # Step 3.5: Return diagnosis metrics and flagged call indices
    return dict(
        modal_fs=modal_fs,
        odd_rate=np.flatnonzero(fs != modal_fs),                 # Calls with atypical sample rates
        median=med,
        std=sd,
        clip_frac=clip,
        offset_calls=np.flatnonzero(np.abs(med) > 0.25 * sd),   # Calls where DC offset > 25% std
        clipped_calls=np.flatnonzero(clip > 0.005)              # Calls with > 0.5% clipped samples
    )


def clean_call(x, remove_offset=True):
    """
    Preprocess raw call audio by casting to float64 and removing DC bias.

    Parameters:
        x (np.ndarray): Raw int16 audio waveform.
        remove_offset (bool): If True, subtract the median to center the signal around 0.

    Returns:
        np.ndarray: Centered float64 audio waveform.
    """
    # Step 3.6: Convert to float64 to avoid overflow during filtering/FFT
    x = x.astype(np.float64)

    # Step 3.7: Center waveform by subtracting median (DC offset removal)
    out = x - np.median(x) if remove_offset else x
    assert out.shape == x.shape

    return out


def fix_effects(calls, fs, diag, baseline_keys):
    """
    Perform an ablation study: re-decode calls without each preprocessing fix
    to quantify how data cleaning impacts decoding accuracy and business metrics.

    Parameters:
        calls (list of np.ndarray): Segmented call waveforms.
        fs (np.ndarray): Per-call sampling rates.
        diag (dict): Diagnostic results from diagnose().
        baseline_keys (list of str): Cleanly decoded key strings with all fixes applied.

    Returns:
        dict: Comparison metrics showing which calls changed and the degraded business summaries.
    """
    # Step 3.8: Re-decode without DC offset removal
    no_off = [decode_call(c, int(r), remove_offset=False)["keys"] for c, r in zip(calls, fs)]

    # Step 3.9: Re-decode assuming a uniform (modal) sample rate for all calls (ignoring per-call fs)
    wrong_fs = [
        decode_call(c, diag["modal_fs"] if i in diag["odd_rate"] else int(r))["keys"]
        for i, (c, r) in enumerate(zip(calls, fs))
    ]

    # Step 3.10: Identify calls whose decoded strings were altered by omitting each fix
    out = {}
    for name, ks in (("offset removal", no_off), ("the per-call sample rate", wrong_fs)):
        changed = [i for i, (a, b) in enumerate(zip(baseline_keys, ks)) if a != b]
        out[name] = dict(calls_changed=changed, summary=summarise(ks))

    return out



In [ ]:
# Analysis

def detect_keys(S, k=6.0):
    """Frame-level key detection.
    S (frames, 7) -> key index (frames,) in 0..11, or -1 when no tone.
    A frame holds a tone if the strongest ROW frequency AND the strongest COLUMN
    frequency are both > k x the call's background level.
    Background = median over frames of the per-frame median of the 7 strengths
    (a frame has at most 2 tone bins, so the 4th-largest value is always background).
    """
    assert S.ndim == 2 and S.shape[1] == 7
    rows, cols = S[:, :4], S[:, 4:]
    floor = max(np.median(np.median(S, axis=1)), 1e-6)
    active = (rows.max(1) > k * floor) & (cols.max(1) > k * floor)
    idx = np.where(active, rows.argmax(1) * 3 + cols.argmax(1), -1)
    assert idx.shape == (S.shape[0],)
    return idx


def runs_to_presses(idx, min_frames=3):
    """Collapse runs of identical consecutive frames into one key press.
    idx (frames,) -> key_index (p,), length_in_frames (p,)
    Runs shorter than min_frames are transition glitches and are dropped.
    A held key is ONE long run, so it stays one press.
    """
    if idx.size == 0:
        return np.empty(0, int), np.empty(0, int)
    change = np.flatnonzero(idx[1:] != idx[:-1]) + 1
    starts = np.r_[0, change]
    lens = np.diff(np.r_[starts, idx.size])
    keep = (idx[starts] >= 0) & (lens >= min_frames)
    return idx[starts][keep], lens[keep]


def decode_call(x_int, fs, remove_offset=True):
    """(len,) int16, fs -> dict(S (frames,7), idx (frames,), keys str, lens (p,))"""
    x = clean_call(x_int, remove_offset)
    if x.size < int(fs * FRAME_MS / 1000):
        return dict(S=np.empty((0, 7)), idx=np.empty(0, int), keys="", lens=np.empty(0, int))
    S = tone_strengths(x, fs, FRAME_MS, HOP_MS)
    idx = detect_keys(S)
    p, lens = runs_to_presses(idx)
    return dict(S=S, idx=idx, keys="".join(KEYS_FLAT[p]), lens=lens)


Business Answers

In [ ]:
def interpret_call(keys):
    """Apply the menu rules to one key string (a small state machine: each key's meaning
    depends on the state, so this part is inherently sequential; 30 short strings only).
    MAIN : 1-4 -> sub-menu, 0 -> agent (end)
    SUB  : 9 -> back to MAIN, 0 -> agent (end), other digit -> option chosen (end),
           except in Transfers (2) where it starts AMOUNT entry
    AMOUNT: every digit is part of the amount (0 is NOT 'agent' here), '#' ends
    """
    state, menu_id, first, agent = "MAIN", None, None, False
    outcome, service, amount = "hung_up", None, ""
    for ch in keys:
        if state == "MAIN":
            if first is None:
                first = ch
            if ch == "0":
                agent, outcome = True, "agent"; break
            if ch in "1234":
                menu_id, state = ch, "SUB"
        elif state == "SUB":
            if ch == "9":
                state = "MAIN"
            elif ch == "0":
                agent, outcome = True, "agent"; break
            elif ch.isdigit():
                service = (menu_id, ch)
                if menu_id == "2":
                    state = "AMOUNT"
                else:
                    outcome = "option_chosen"; break
        else:  # AMOUNT
            if ch == "#":
                outcome = "transfer_completed"; break
            if ch.isdigit():
                amount += ch
    return dict(first=first, agent=agent, outcome=outcome, service=service,
                amount=int(amount) if amount and outcome == "transfer_completed" else None)


def summarise(key_strs):
    """list of n_calls key strings -> counts for the two business questions."""
    res = [interpret_call(s) for s in key_strs]
    first = np.array([r["first"] or "none" for r in res])
    agent = np.array([r["agent"] for r in res])
    outcome = np.array([r["outcome"] for r in res])
    f_vals, f_cnt = np.unique(first, return_counts=True)
    o_vals, o_cnt = np.unique(outcome, return_counts=True)
    assert f_cnt.sum() == len(key_strs) == o_cnt.sum()
    return dict(per_call=res, first=dict(zip(f_vals.tolist(), f_cnt.tolist())),
                outcome=dict(zip(o_vals.tolist(), o_cnt.tolist())),
                agent_calls=int(agent.sum()))


def make_charts(calls, fs, decoded, summ, menu, out_dir):
    import os
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)

    # Chart 1: what the decoder sees in one call
    i = int(np.argmax([len(d["keys"]) for d in decoded]))
    S, idx = decoded[i]["S"], decoded[i]["idx"]
    t = np.arange(S.shape[0]) * HOP_MS / 1000
    fig, ax = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True,
                           gridspec_kw=dict(height_ratios=[3, 1.4]))
    for j, f in enumerate(FREQS):
        ax[0].plot(t, S[:, j], lw=1, label=f"{int(f)} Hz")
    ax[0].set_ylabel("tone strength"); ax[0].legend(ncol=7, fontsize=6, loc="upper right")
    ax[0].set_title(f"Call {i}: tone strengths and decoded keys")
    ax[1].step(t, idx, where="post", color="k", lw=1)
    ax[1].set_yticks(range(12)); ax[1].set_yticklabels(KEYS_FLAT, fontsize=6)
    ax[1].set_ylabel("key"); ax[1].set_xlabel("time (s)")
    fig.tight_layout(); fig.savefig(f"{out_dir}/chart1_tones_and_keys.png", dpi=150); plt.close(fig)

    # Chart 2: services + outcomes
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
    order = ["1", "2", "3", "4", "0"]
    ax[0].bar([MAIN_NAMES[k] for k in order], [summ["first"].get(k, 0) for k in order],
              color="#2a6f97")
    ax[0].set_title("First main-menu choice"); ax[0].set_ylabel("callers")
    ax[0].tick_params(axis="x", labelrotation=30)
    o = summ["outcome"]
    ax[1].bar(list(o.keys()), list(o.values()), color="#52b788")
    ax[1].set_title(f"How calls end (agent asked at any point: {summ['agent_calls']})")
    ax[1].tick_params(axis="x", labelrotation=30)
    fig.tight_layout(); fig.savefig(f"{out_dir}/chart2_services_outcomes.png", dpi=150); plt.close(fig)